In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import torch
import gc

In [68]:
# clearing GPU cache:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [69]:
# force garbage collection:
gc.collect()
print('cache cleared and garbage collected!')

cache cleared and garbage collected!


In [24]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [25]:
import os, sys
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [27]:
from load_weights import gpt as model
print("model loaded successfully.")

model loaded successfully.


In [28]:
gc.collect()
torch.cuda.empty_cache()

In [29]:
import tiktoken, json
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

In [30]:
class JsonInstructionDataset(Dataset):
    def __init__(self, json_data, max_length=512,):
        self.encoding = tiktoken.get_encoding('gpt2')
        self.json_data = json_data
        self.max_length = max_length

    def __len__(self):
        return len(self.json_data)

    def __getitem__(self, idx):
        item = self.json_data[idx]

        text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        # tokenizing with tiktoken
        tokens = self.encoding.encode(text)

        # truncating if necessary
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        input_ids = tokens

        # padding if necessary
        if len(input_ids) < self.max_length:
            padding_length = self.max_length - len(input_ids)
            padding = [self.encoding.eot_token] * padding_length
            input_ids = input_ids + padding

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(input_ids, dtype=torch.long)
        }

In [31]:
import json

In [32]:
with open("/content/drive/MyDrive/llm_from_scratch/datasets/alpaca_gpt4_data.json", "r", encoding="utf-8") as f:
    json_data_gpt4 = json.load(f)

In [33]:
print(len(json_data_gpt4))

52002


In [56]:
json_gpt4_train, json_gpt4_val = json_data_gpt4[5000:10000], json_data_gpt4[10000:10500]
print(len(json_gpt4_train))
print(len(json_gpt4_val))

5000
500


In [57]:
print(json_gpt4_train[-1])
print(json_gpt4_val[-2])

{'instruction': 'Document the steps needed to deploy a machine learning model in an Android application.', 'input': '', 'output': "There are various approaches, tools and frameworks available for deploying a machine learning model in an Android application. One such approach, using TensorFlow Lite, is described below.\n\n1. Select or train a machine learning model: The first step in deploying a machine learning model in an Android application is to select or train a model that can solve the required task. This can be done using popular machine learning libraries such as TensorFlow, PyTorch, or scikit-learn.\n\n2. Convert the machine learning model to TensorFlow Lite format: Android applications use the TensorFlow Lite framework for running machine learning models on devices. To make the trained machine learning model compatible with TensorFlow Lite, it needs to be converted into the TensorFlow Lite format using the TensorFlow Lite Converter.\n\n3. Add TensorFlow Lite library to the And

In [58]:
train_dataset = JsonInstructionDataset(json_gpt4_train)
val_dataset = JsonInstructionDataset(json_gpt4_val)

In [59]:
print(train_dataset[0])
print("successfully printed.")

{'input_ids': tensor([21017, 46486,    25,   198,  8645,   378,   257, 19780,  3526,   351,
          262,  1813,  2546,   290,   351,  5207,   379,   262,  7368,  6116,
           13,   198,   198, 21017, 23412,    25,   198, 10699,    25,   807,
           87,    23,   220,   198,    47,  8535,  6116,    25,   198, 12256,
          371,   566,   379,   367,    23,   198, 12256,   350,  3832,   379,
          347,    18,   198,  9915,   350,  3832,   379,   317,    18,   198,
         9915,   371,   566,   379,   367,    17,   198,   198, 21017, 18261,
           25,   198,  4342,   318,   262,   807,    87,    23, 19780,  3526,
          351,   262,  7368,  5207,   379,   262, 10348,  6116,    25,   198,
          198, 15506,    63,   198,    23,   220,   220, 20724,   250,   764,
          764,   764,   764,   764,   764,   764,   220,   198,    22,   220,
          220,   764,   764,   764,   764,   764,   764,   764,   764,   220,
          198,    21,   220,   220,   764,   764, 

In [60]:
from transformers import PretrainedConfig

class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [61]:
import types

def hf_forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
    batch_size, seq_len = input_ids.shape

    with torch.amp.autocast('cuda', enabled=True):
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

    loss = None
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    return (loss, logits) if loss is not None else logits

model.forward = hf_forward.__get__(model, type(model))

In [62]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [63]:
from peft import LoraConfig, get_peft_model

# model = model.half()

def setup_lora_model(model):

    target_modules = [
        "W_query", "W_key", "W_value",
        "out_proj",
        "out_head"
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        modules_to_save=None
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [70]:
import transformers

transformers.logging.set_verbosity_info()

training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",

    num_train_epochs=1,
    learning_rate=3e-5,

    # Batch size and gradient settings
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimization settings
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Mixed precision training
    fp16=True,

    # Evaluation settings
    eval_strategy="steps",
    eval_steps=30,

    # Data loading settings
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,

    # Progress and logging
    logging_steps=30,
    disable_tqdm=False,
    report_to="none",

    # Save settings
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1
)

PyTorch: setting up devices


In [71]:
def compute_metrics(eval_pred):
    return {}

In [72]:
os.environ["WANDB_DISABLED"] = "true"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

Using auto half precision backend


In [73]:
gc.collect()
torch.cuda.empty_cache

print("Started training...")
trainer.train()
print("Training completed!")

Started training...


***** Running training *****
  Num examples = 5,000
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 313
  Number of trainable parameters = 3,361,416


Step,Training Loss,Validation Loss
5,57.980200,7.379098
10,60.871600,7.313740
15,54.260000,7.200383
20,51.263800,6.961582



***** Running Evaluation *****
  Num examples = 500
  Batch size = 2

***** Running Evaluation *****
  Num examples = 500
  Batch size = 2

***** Running Evaluation *****
  Num examples = 500
  Batch size = 2

***** Running Evaluation *****
  Num examples = 500
  Batch size = 2


KeyboardInterrupt: 

In [ ]:
# def test_model(model, instruction, input_text="", max_new_tokens=1024):
#     # Format the prompt like during training
#     if input_text:
#         prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
#     else:
#         prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

#     # Tokenize
#     encoding = tiktoken.get_encoding('gpt2')
#     input_ids = encoding.encode(prompt)
#     input_tensor = torch.tensor([input_ids]).to(device)

#     # Manual generation (no .generate() method)
#     model.eval()
#     with torch.no_grad():
#         generated = input_tensor

#         for i in range(max_new_tokens):
#             # Forward pass
#             # Access logits directly from the returned tensor
#             logits = model(input_ids=generated)

#             # Get last token logits
#             next_token_logits = logits[0, -1, :]

#             # Apply temperature and sample
#             next_token_logits = next_token_logits / 0.7  # temperature
#             probs = torch.softmax(next_token_logits, dim=-1)
#             next_token = torch.multinomial(probs, num_samples=1)

#             # Stop if EOT token is generated BEFORE appending
#             if next_token.item() == encoding.eot_token:
#                 break

#             # Append to generated sequence
#             generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

#     # Decode
#     response = encoding.decode(generated[0].tolist())
#     # Extract only the response part (after "### Response:\n")
#     response_text = response.split("### Response:\n")[-1]

#     # Remove endoftext token if it exists and clean up
#     response_text = response_text.replace('<|endoftext|>', '').strip()

#     return response_text

# # Test examples
# print("🧪 Testing the fine-tuned model:\n")

# # Example 1: General instruction
# test1 = test_model(
#     model,
#     instruction="how can i learn english faste explain in points.",
#     input_text="",
# )
# print(f"Test 1 - Explanation:\n{test1}\n")

In [ ]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/instruction_tuned_model_weights.pth")

# print("✅ Saved unified model with instruction knowledge")